In [12]:
import pandas as pd, numpy as np
import json, os

In [13]:
from jh_utils.time_series.time_dataframe import time_series_dataframe

In [14]:
## read dataset
df = pd.read_csv('../../data/03_processed/processed.csv')

## Na from dataset
na_value = -9999

## organize index and drop hour and dates
df['date_time'] = df['data']+' '+df['hora']
df['date_time'] = pd.to_datetime(df['date_time'])
df = df.rename(columns={'hora': 'hour'})
df.hour = df.hour.apply(lambda x: x.split(':')[0])
## set an index
df.index = df['date_time']
df = df.drop(columns=['data','date_time','hour'])
df = df.replace(to_replace=na_value,value=np.NaN)

## fill na with previous values
df = df.fillna(method='ffill')
df = df.reset_index()
df = df[df.date_time>'2006-11-01']
df.reset_index(drop=True, inplace=True)
df.index = df.date_time

In [15]:
## ! concatenate 
time_df = time_series_dataframe('2006-11-01','2021-01-01').iloc[1:]
df = pd.concat([df,time_df],axis=1).iloc[:,1:]

In [18]:
df.index.name = 'datetime'

In [50]:
datasets = '../../src/d01_data/datasets.json'
with open(datasets) as data:
    datasets = json.load(data)

dataset_keys = list(datasets.keys())

In [51]:
def make_model_dataframe(df, dataset, fixed_columns = ['hour', 'month', 'weekday', 'sin', 'cos']):
    ## filtering the date
    df = df[df.index > datasets[dataset]['start_date']]
    ## declaring the output-dataframe
    df_out = pd.DataFrame()
    ## filtering regex
    for i in datasets[dataset]['regex_columns'] + fixed_columns:
        df_out = pd.concat([df_out,df.filter(regex=i)],axis=1)
    return df_out

In [55]:
for i in dataset_keys:
    make_model_dataframe(df,i).to_csv('../../data/04_datasets/{dataset}.csv'.format(dataset = i),index = False)